In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Case Study: Optimizing a Transformer

Each section of this chapter has isolated one cause of a slow training step and the change that removes it: host dispatch overhead, memory traffic from small kernels, wide number formats, and single-GPU boundaries. A real training step suffers from several at once, and removing one shifts the bottleneck to the others. Here we take a single model, as a practitioner would, and ask how much faster its training step can become—and in what order the changes should be applied.

The model is the GPT of that section with its width doubled to 512 (19 million parameters). One training step on 64 windows of 128 bytes ($8{,}192$ tokens) performs about $6 \times 1.9 \times 10^{7} \times 8{,}192 \approx 9 \times 10^{11}$ FLOPs: two per parameter and token in the forward pass and four in the backward pass. On an RTX 4090 ($165\text{ TFLOP/s}$ in bf16), the arithmetic needs less than $6\text{ ms}$. A plain fp32 step with AdamW takes about $25\text{ ms}$. Half of that gap comes from the number format alone (tf32 tensor cores run at half the bf16 rate, requiring $11\text{ ms}$); the other half is time in which the tensor cores run below peak or sit idle.

We follow the method of that section: count, predict, measure, and explain the difference. Before each change, the profile of the previous step bounds what the change can gain; after it, we compare. Each change is measured cumulatively on top of the previous ones. The section concludes by checking that the fastest step still trains the model and accounting for where its remaining time goes.

In [ ]:
# Let XLA overlap collectives with computation. JAX reads XLA_FLAGS once,
# when it starts, so this cell runs before the imports. Our outputs come
# from a run with three more settings in the environment, which you set
# before starting: --xla_gpu_enable_triton_gemm=false in XLA_FLAGS, so that
# XLA calls cuBLAS for the matrix products (the Triton kernels it would
# generate run the products of the large minibatches more slowly on the
# RTX 4090); NCCL_LOCAL_REGISTER=0, since NCCL's buffer registration fails
# on GPUs without peer-to-peer access and would print a warning at every
# collective; and XLA_PYTHON_CLIENT_ALLOCATOR=vmm, which maps each array
# into one contiguous range: the fp32 step of the training check needs
# about 13 GiB in one piece after many other arrays have come and gone.
import os
os.environ['XLA_FLAGS'] = (os.environ.get('XLA_FLAGS', '') +
                           ' --xla_gpu_enable_latency_hiding_scheduler=true')

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import re
from jax.sharding import NamedSharding, PartitionSpec as P

## Experimental Setup

Every change in this section is judged by the same measurement, so the model, data, and throughput benchmark stay fixed throughout.

### Model and Data

We train `d2l.GPT` (that section): six pre-norm blocks with eight attention heads each, rotary position embeddings, RMSNorm, SwiGLU, and an input embedding tied to the output layer. The data are 128-byte windows of WikiText-103 drawn at random offsets over the 256-byte vocabulary (`d2l.WikiText103`, that section).

We double the width from 256 to 512 so the matrix products sit comfortably in the compute-bound regime (that section). Multiplying an $n \times d$ activation matrix by a $d \times d'$ weight matrix performs $2ndd'$ FLOPs and moves $b(nd + dd' + nd')$ bytes ($b$ bytes per element). For $n = 8{,}192$ tokens at width $d = 512$, the block projections reach $125\text{--}185\text{ FLOP/byte}$ in tf32 ($b = 4$) and $250\text{--}370\text{ FLOP/byte}$ in bf16 ($b = 2$), well above the RTX 4090 ridge points ($82\text{ FLOP/byte}$ for tf32 and $164\text{ FLOP/byte}$ for bf16). A faster number format therefore shortens these products directly. At width 256 they straddle the ridge point (Exercise 6).

The 270 MB training corpus fits in GPU memory as a flat byte array. The next cell places it on the GPU, defines `windows` to sample minibatches directly on the device without host transfers, and builds the model.

In [ ]:
data = d2l.WikiText103(batch_size=64, num_steps=128)
corpus = jnp.asarray(data.train)  # One byte per token

def windows(corpus, batch_size, num_steps=128, *, sharding=None):
    """Endless minibatches of windows at random offsets, each position with
    the next byte as its target; `sharding` spreads them over devices."""
    offsets = jnp.arange(num_steps + 1)
    while True:
        starts = jax.random.randint(d2l.get_key(), (batch_size, 1), 0,
                                    len(corpus) - num_steps)
        if sharding is not None:
            starts = jax.device_put(starts, sharding)
        W = corpus[starts + offsets].astype(jnp.int32)
        yield W[:, :-1], W[:, 1:]

model = d2l.GPT(len(data.vocab), num_hiddens=512,
                rngs=nnx.Rngs(d2l.get_key()))
num_params = d2l.num_params(model)
print(f'{num_params / 1e6:.1f}M parameters, '
      f'{len(corpus) / 1e6:.0f}M training bytes on the GPU')

At 16 bytes per parameter (weight, gradient, and two Adam moments), the 19M-parameter training state occupies only $0.3\text{ GiB}$, requiring neither gradient accumulation nor sharding (that section, that section). Keeping the corpus on the GPU also isolates the training step from the input pipeline (that section).

### Throughput Measurement

We measure end-to-end training throughput in tokens per second, including minibatch sampling, forward and backward passes, and the optimizer update:

| setting | value |
|:--|:--|
| hardware | 1 of 4 RTX 4090 GPUs (2 and 4 for data parallelism) |
| model | `d2l.GPT`: 6 blocks, width 512, 8 heads, byte vocabulary, context 128 |
| data | random 128-byte windows of WikiText-103 drawn on the GPU every step |
| optimizer | AdamW ($\text{lr} = 10^{-3}$, weight decay $0.01$) |
| precision | fp32 with tf32 matrix products until bf16 is introduced |
| timing | `d2l.Benchmark`: 20 warm-up steps, then 3 windows of 50 steps between device syncs |
| result | median tokens/s across the 3 windows, ratio to previous kept config, and relative window spread |

`tokens_per_second` applies these rules via `d2l.Benchmark` (that section). `record` stores each result and prints its speedup over the previous kept configuration along with the relative spread across the three measurement windows ($\text{range} / \text{median}$). Configurations passed with `kept=False` are stored as branches that subsequent configurations do not build upon.

In [ ]:
def tokens_per_second(step, batches, warmup=20, repeats=50, num_windows=3):
    """Median training throughput of `step` on minibatches drawn from
    `batches`, over `num_windows` timed windows of `repeats` steps each, and
    the spread of the windows (largest minus smallest) relative to it."""
    X, _ = next(batches)
    speeds = [math.prod(X.shape) / d2l.Benchmark(
                  lambda: step(*next(batches)), warmup if i == 0 else 0,
                  repeats).time for i in range(num_windows)]
    return np.median(speeds), np.ptp(speeds) / np.median(speeds)

results, branches = {}, {}
def record(name, measured, kept=True):
    """Store a measured throughput and print it with its ratio to the last
    kept configuration and the spread of its windows; a configuration that
    is not kept is stored in `branches`."""
    speed, spread = measured
    previous = list(results.values())[-1] if results else speed
    (results if kept else branches)[name] = speed
    print(f'{name}: {speed:,.0f} tokens/s ({speed / previous:.2f}x), '
          f'windows within {spread:.1%}')

Steps that perform substantial host work are sensitive to background host activity and show a wider window spread than compiled or graph-captured steps. Because day-to-day host load varies more than consecutive windows in a single run, we read speedup ratios to one decimal place and trust small gains of a few percent only when the profile predicts them.

## A Single GPU

We first optimize the training step on a single GPU, one change at a time, starting from the baseline implementation of that section.

### Baseline

The baseline is the standard training step of that section with AdamW: forward pass, cross-entropy loss, backward pass, and parameter update.

`train_step` is compiled with `nnx.jit` as in `d2l.train_steps` (that section). It takes the model and optimizer as arguments and computes the cross-entropy loss from fp32 logits.

In [ ]:
def train_step(model, optimizer, X, Y):
    """Loss, gradients, and AdamW update for one minibatch."""
    def loss_fn(model):
        logits = model(X).astype(jnp.float32)
        return optax.softmax_cross_entropy_with_integer_labels(
            logits, Y).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

adamw = optax.adamw(1e-3, weight_decay=0.01)
optimizer = nnx.Optimizer(model, adamw, wrt=nnx.Param)
jit_step = nnx.jit(train_step)
step = lambda X, Y: jit_step(model, optimizer, X, Y)
record('baseline', tokens_per_second(step, windows(corpus, 64)))

As expected, the baseline windows spread roughly ten times more widely than later configurations because of per-step host work. To see where the step spends its time, `profile` records five steps with `d2l.kernel_times`, displays the breakdown with `d2l.show_profile` (that section)—total device kernel time versus wall-clock step time, the share taken by matrix products (including fused attention kernels), and the top kernels—and saves the profile for our final accounting.

In [ ]:
profiles = {}
def profile(step, batches):
    """Kernel profile of `step`, the configuration recorded last, on
    minibatches drawn from `batches`: printed against the throughput that
    `record` measured, and stored in `profiles` under the same name."""
    name, speed = list(results.items())[-1]
    X, _ = next(batches)
    kernels = d2l.kernel_times(lambda: step(*next(batches)), steps=5)
    d2l.show_profile(kernels, 1e3 * math.prod(X.shape) / speed, rows=4)
    profiles[name] = kernels

profile(step, windows(corpus, 64))

The device runs kernels for nearly the entire step, with matrix products taking about two-thirds of the time. Because the host mostly keeps up, the step is bottlenecked by device work.

Because `nnx.jit` compiles the entire step, XLA has already fused the elementwise operations into surrounding kernels. The third of device time outside the matrix products goes to these fused kernels, which remain limited by the bytes they move.

### Host Overhead and Small Kernels

Two remedies target the time spent outside matrix products, and the profile bounds what each can gain:

1. *Reducing host overhead* targets idle gaps where the device waits for the host (that section); its speedup is bounded by the idle fraction of the step.
2. *Kernel fusion* merges chains of bandwidth-bound elementwise kernels so intermediate arrays stay on chip; its speedup is bounded by the non-matmul device time.

Our step is already fused by XLA and the GPU waits for only 2% of the step, so reducing host work can gain at most 2% right now. We remove it anyway because it is a fixed per-step cost that would surface once later optimizations shorten device execution. On every call, `nnx.jit` traverses the Python model and optimizer objects to extract and update their arrays. Below, we benchmark this traversal overhead on a no-op function using plain `nnx.jit`, `nnx.jit(..., graph=False)` (tree mode), and `nnx.jit_partial(..., graph=False)` (which binds the model and optimizer state once so each step passes only the minibatch).

In [ ]:
print(f'{len(jax.tree.leaves(nnx.state((model, optimizer))))} arrays in '
      'the model and the optimizer')
noop = lambda model, optimizer: None
walk, tree = nnx.jit(noop), nnx.jit(noop, graph=False)
bound = nnx.jit_partial(noop, model, optimizer, graph=False)
for name, call in (('nnx.jit', lambda: walk(model, optimizer)),
                   ('nnx.jit, graph=False', lambda: tree(model, optimizer)),
                   ('nnx.jit_partial', bound)):
    seconds = d2l.Benchmark(call, repeats=100).time
    print(f'{name}: {1e3 * seconds:.2f} ms per call that does nothing')

A no-op call through `nnx.jit` takes nearly $9\text{ ms}$ of host time—over a third of the step's device time! That cost is hidden only as long as the host stays ahead of the device; switching to tree traversal removes most of it, and binding with `nnx.jit_partial` eliminates nearly all the rest. Next, we verify that `nnx.jit_partial` produces identical loss and weight updates to `nnx.jit`.

In [ ]:
twin = nnx.clone((model, optimizer))  # Equal states for the check
step = nnx.jit_partial(train_step, model, optimizer, graph=False)
X, Y = next(windows(corpus, 64))
loss, twin_loss = step(X, Y), jit_step(*twin, X, Y)
gaps = jax.tree.map(lambda a, b: jnp.abs(a - b).max(),
                    nnx.state(model, nnx.Param), nnx.state(twin[0], nnx.Param))
gap = max(float(g) for g in jax.tree.leaves(gaps))
print(f'losses {loss:.4f} and {twin_loss:.4f}, largest difference of the '
      f'updated weights {gap:.1e}')
assert gap < 1e-6
record('+ jit_partial', tokens_per_second(step, windows(corpus, 64)))

The updated weights agree exactly, and throughput rises by about 1%, within the 2% bound and the baseline noise spread.

A second profile shows how the step changed:

In [ ]:
profile(step, windows(corpus, 64))

The device remains busy for nearly the entire step, and matrix products now account for roughly two-thirds or more of device time.

Device kernel times are unchanged because `nnx.jit_partial` only reduced host-side dispatch overhead.

### Reduced Precision

With the GPU busy and matrix products dominating device time, the next step is to speed up the matrix products. Switching from tf32 to bf16 doubles RTX 4090 tensor-core peak throughput to $165\text{ TFLOP/s}$ and halves the memory footprint of bf16 activations (that section). If matrix products take a fraction $f \in [2/3, 0.7]$ of the step and only they double in speed, the step shrinks to $1 - f/2$ of its time—a speedup of about $1.5\times$ (rising toward $2\times$ if non-matmul kernels also move half as many bytes).

We apply the mixed-precision recipe of that section: matrix products (including attention) and intermediate elementwise operations run in bf16, while master weights, gradients, Adam moments, normalizations, the loss, and the residual stream remain in fp32 (keeping the residual stream in fp32 prevents small block updates from rounding away in bf16's 8-bit significand). Because bf16 matches fp32's exponent range, no loss scaling is needed. Before timing, we verify that the bf16 forward pass matches fp32 logits to within bf16 precision (~1%).

`d2l.to_bf16` (that section) sets `dtype=jnp.bfloat16` on the linear layers while leaving `param_dtype` and all other layers (embeddings, normalizations, and the tied output projection) in fp32.

In [ ]:
xla16 = d2l.GPT(len(data.vocab), num_hiddens=512,
                rngs=nnx.Rngs(d2l.get_key()))
nnx.update(xla16, nnx.state(model, nnx.Param))  # The same weights
d2l.to_bf16(xla16)
X, _ = next(windows(corpus, 64))
high = model(X)
error = float(jnp.linalg.norm(xla16(X) - high) / jnp.linalg.norm(high))
print(f'relative difference of the bf16 logits: {error:.1e}')
assert error < 0.01
step = nnx.jit_partial(train_step, xla16,
                       nnx.Optimizer(xla16, adamw, wrt=nnx.Param), graph=False)
record('+ bf16', tokens_per_second(step, windows(corpus, 64)))

The bf16 logits agree with fp32 to within 1%.

Throughput rises by $1.45\times$, just under the $1.48\times$ predicted by halving the 65% matrix-product share alone, indicating that non-matmul kernels did not speed up. One culprit is attention: by default, `jax.nn.dot_product_attention` uses XLA's implementation, which materializes the full attention matrix in device memory (that section). Now that activations are in 16-bit precision, we can switch to cuDNN's fused FlashAttention kernel (`implementation='cudnn'`), which keeps the attention scores on chip.

In [ ]:
class FusedGPT(d2l.GPT):
    """d2l.GPT with cuDNN's fused attention."""
    class CausalAttention(d2l.GPT.CausalAttention):
        @staticmethod
        def attend(q, k, v):  # cuDNN takes 16-bit inputs only
            return jax.nn.dot_product_attention(q, k, v, is_causal=True,
                                                implementation='cudnn')

In [ ]:
model16 = FusedGPT(len(data.vocab), num_hiddens=512,
                   rngs=nnx.Rngs(d2l.get_key()))
nnx.update(model16, nnx.state(model, nnx.Param))  # The same weights
d2l.to_bf16(model16)
error = float(jnp.linalg.norm(model16(X) - high) / jnp.linalg.norm(high))
print(f'relative difference of the bf16 logits: {error:.1e}')
assert error < 0.01
step = nnx.jit_partial(train_step, model16,
                       nnx.Optimizer(model16, adamw, wrt=nnx.Param),
                       graph=False)
record('+ fused attention', tokens_per_second(step, windows(corpus, 64)))

Switching to cuDNN fused attention raises throughput by another third while preserving 1% logit agreement, bringing the combined bf16 + fused-attention speedup close to $2\times$.

### Larger Minibatches

Increasing the minibatch size amortizes fixed per-step costs over more tokens. Because the device is already busy throughout the step, only device work that scales with parameter count rather than token count can be amortized:

- AdamW reads weights, gradients, and two moments, and writes weights and two moments ($28\text{ bytes/param}$ in fp32).
- Mixed precision casts fp32 master weights to bf16 in the forward pass and bf16 weight gradients back to fp32 in the backward pass ($12\text{ bytes/param}$).

Together, these move $40\text{ bytes/param}$, or $0.76\text{ GB/step}$. At $1.0\text{ TB/s}$ of memory bandwidth, this fixed work takes about $0.75\text{ ms}$—roughly $1/16$ of a $12\text{--}13\text{ ms}$ bf16 step at batch size 64. Growing the minibatch $8\times$ from 64 to 512 windows ($65{,}536$ tokens) amortizes $7/8$ of this fixed cost, predicting around a 5% throughput gain.

Even a modest single-GPU gain is worthwhile because data parallelism scales well only when the per-GPU step time is much longer than the gradient allreduce (that section). Keep in mind, however, that larger batches reduce sample efficiency once the batch size exceeds the critical batch size (that section, Exercise 5). Because activation memory grows linearly with batch size, we also record peak memory beyond weights and optimizer state (that section).

In [ ]:
batches = windows(corpus, 512)
plan = step.lower(*next(batches)).compile().memory_analysis()
record('+ batch 512', tokens_per_second(step, batches))
memory = plan.temp_size_in_bytes / 2**30
print(f'memory beyond weights and optimizer state: {memory:.1f} GiB')

At batch size 512, activation memory uses about a third of the GPU's $24\text{ GiB}$, and throughput increases by a few percent (Exercise 2 separates the fixed per-step cost from the per-token cost across batch sizes). A third profile shows the breakdown at batch size 512:

In [ ]:
profile(step, windows(corpus, 512))

The device is busy for the entire step: matrix products (including fused attention) take just under 60% of the time, while fused bandwidth-bound kernels for RMSNorm, SwiGLU, rotary embeddings, and fp32/bf16 casts take the rest.

### Activation Checkpointing

Activation checkpointing (that section) saves memory by storing only each block's input during the forward pass and recomputing its internal activations during the backward pass. Because batch size 512 uses only a third of GPU memory, checkpointing is unnecessary here and can only slow the step down. We can predict the slowdown in two ways:

1. *By FLOP count*: recomputing the forward pass raises per-token block arithmetic from $6$ to $8\text{ FLOPs/param}$, predicting $6/8 = 0.75$ of the uncheckpointed throughput.
2. *By timing the forward pass*: below, we time a standalone forward pass $t_{\text{fwd}}$ at batch size 512 and compute $1 / (1 + t_{\text{fwd}} / t_{\text{step}})$.

In [ ]:
X512, _ = next(windows(corpus, 512))  # One minibatch of the last step
forward = nnx.jit_partial(lambda model, X: model(X), model16, graph=False)
t_forward = d2l.Benchmark(lambda: forward(X512)).time

In [ ]:
t_step = 512 * 128 / results['+ batch 512']  # Seconds per step
print(f'forward pass {1e3 * t_forward:.1f} ms of a {1e3 * t_step:.1f} ms '
      f'step: repeating it keeps at least {1 / (1 + t_forward / t_step):.2f} '
      f'of the throughput')

Both estimates agree: the forward pass takes about a third of the step, so repeating it should retain at least three-quarters of the throughput (slightly more because embeddings and the output projection are not checkpointed). We now wrap each block in `Checkpointed`—using `torch.utils.checkpoint.checkpoint` in PyTorch and `nnx.remat` in JAX—verify logit equality, and measure both throughput (`kept=False`) and activation memory.

In [ ]:
class Checkpointed(nnx.Module):
    """A module that keeps only its input for the backward pass."""
    def __init__(self, module):
        self.module = module

    def __call__(self, X, **kwargs):
        return nnx.remat(lambda module, X, kwargs: module(X, **kwargs))(
            self.module, X, kwargs)

remat = FusedGPT(len(data.vocab), num_hiddens=512,
                 rngs=nnx.Rngs(d2l.get_key()))
nnx.update(remat, nnx.state(model16, nnx.Param))
remat.blks = nnx.List(Checkpointed(blk) for blk in remat.blks)
d2l.to_bf16(remat)
assert jnp.allclose(remat(X), model16(X), atol=1e-6)
optimizer = nnx.Optimizer(remat, adamw, wrt=nnx.Param)
step = nnx.jit_partial(train_step, remat, optimizer, graph=False)
batches = windows(corpus, 512)
plan = step.lower(*next(batches)).compile().memory_analysis()
record('+ checkpointing', tokens_per_second(step, batches), kept=False)
memory = plan.temp_size_in_bytes / 2**30
print(f'memory beyond weights and optimizer state: {memory:.1f} GiB')

Checkpointing cuts activation memory by roughly two-thirds while retaining more throughput than the standalone forward-pass time predicted.

Throughput drops to $0.79\times$ (above the $0.76\times$ bound) because XLA similarly fuses recomputed elementwise work into the backward pass (Exercise 3).

Checkpointing is essential when a model or batch size would otherwise exceed device memory, but here memory is plentiful, so we leave it out of the configurations that follow.

## Data Parallelism

Our fastest single-GPU step leaves three of the machine's four RTX 4090s idle. Before running on multiple GPUs, we use the cost model of that section to predict the data-parallel speedup.

### Predicted Speedup

In data parallelism (that section), each of $k$ GPUs holds a replica of the model, computes gradients on its own minibatch, and sums gradients via an allreduce of size $S$ bytes in $c$ buckets:

$$
t_{\textrm{compute}}(B/k) + c\,\alpha +
\frac{2(k-1)}{k}\,S\,\beta_{\textrm{link}}.
$$

On our PCIe machine without peer-to-peer access, GPU transfers stage through host memory at a bus bandwidth of roughly $2.4\text{ GB/s}$ (that section), or $\beta_{\textrm{link}} \approx 0.42\text{ ns/byte}$. Under weak scaling (fixed per-GPU batch size), per-bucket latency $c\,\alpha$ is under a millisecond and negligible next to tens of milliseconds of step time, simplifying the step time without overlap to:

$$t_{\textrm{step}}(k) \approx t_{\textrm{step}}(1) + t_{\textrm{allreduce}}(k),$$

for a throughput multiplier of $k\, t_{\textrm{step}}(1) / (t_{\textrm{step}}(1) + t_{\textrm{allreduce}}(k))$. Adding a second GPU increases throughput only if $t_{\textrm{allreduce}}(2) < t_{\textrm{step}}(1)$.

For our 19M-parameter GPT, fp32 gradients occupy $S = 76\text{ MB}$, giving $t_{\textrm{allreduce}}(2) \approx S\beta_{\textrm{link}} \approx 31\text{ ms}$ on two GPUs and $t_{\textrm{allreduce}}(4) \approx 1.5\,S\beta_{\textrm{link}} \approx 47\text{ ms}$ on four:

- *Compiled fp32 at batch 64 per GPU* ($t_{\textrm{step}}(1) \approx 22\text{ ms}$): the allreduce takes longer than the step itself! Equation the equation predicts a *slowdown* on 2 GPUs ($\sim 0.85\times$) and only $\sim 1.3\times$ on 4 GPUs.
- *Compiled bf16 at batch 512 per GPU* ($t_{\textrm{step}}(1) \approx 90\text{--}100\text{ ms}$): the step is much longer than the allreduce, predicting $\sim 1.5\times$ on 2 GPUs and $2.6\text{--}2.7\times$ on 4 GPUs.

Because frameworks overlap gradient allreduce buckets with the backward pass (the figure), the equation serves as a lower bound on speedup; the margin above it measures how much communication was hidden behind backward computation.

### Measured Speedup

`data_parallel(k, bf16, batch_size)` measures training throughput on $k$ GPUs alongside the standalone allreduce time for a $76\text{ MB}$ gradient buffer on the same GPUs.

JAX drives all $k$ GPUs from a single process (that section), replicating the model, optimizer, and corpus across an `Auto` mesh while sharding each $k \times \text{batch\_size}$ minibatch across `'data'`. XLA automatically inserts the gradient allreduce and overlaps it with the backward pass via the latency-hiding scheduler enabled in the first cell.

In [ ]:
def data_parallel(k, bf16, batch_size):
    """Tokens per second on k GPUs, and seconds per allreduce of all
    gradients."""
    mesh = jax.make_mesh((k,), ('data',), devices=jax.devices()[:k],
                         # Auto: the compiler places intermediate results
                         axis_types=(jax.sharding.AxisType.Auto,))
    copy, split = NamedSharding(mesh, P()), NamedSharding(mesh, P('data'))
    net = (FusedGPT if bf16 else d2l.GPT)(len(data.vocab), num_hiddens=512,
                                          rngs=nnx.Rngs(d2l.get_key()))
    net = d2l.to_bf16(net) if bf16 else net
    optimizer = nnx.Optimizer(net, adamw, wrt=nnx.Param)
    nnx.update((net, optimizer),
               jax.device_put(nnx.state((net, optimizer)), copy))
    step = nnx.jit_partial(train_step, net, optimizer, graph=False)
    batches = windows(jax.device_put(corpus, copy), k * batch_size,
                      sharding=split)
    speed, _ = tokens_per_second(step, batches)
    grads = jnp.zeros((k, num_params), device=split)  # All gradients per GPU
    psum = jax.jit(jax.shard_map(lambda g: jax.lax.psum(g, 'data'),
                                 mesh=mesh, in_specs=P('data'),
                                 out_specs=P('data')))
    return speed, d2l.Benchmark(lambda: psum(grads), repeats=10).time

In [ ]:
for bf16, batch_size in ((False, 64), (True, 512)):
    tokens = batch_size * 128  # Per GPU and step
    one, _ = data_parallel(1, bf16, batch_size)
    print(f'{"bf16" if bf16 else "fp32"}, batch {batch_size} per GPU: '
          f'{one:,.0f} tokens/s on one GPU')
    for k in (2, 4):
        if k <= d2l.num_gpus():
            speed, allreduce = data_parallel(k, bf16, batch_size)
            predicted = k * tokens / (tokens / one + allreduce)
            print(f'  {k} GPUs: allreduce {1e3 * allreduce:3.0f} ms, '
                  f'speedup predicted {predicted / one:.2f}x, '
                  f'measured {speed / one:.2f}x')

The standalone allreduce takes $31\text{ ms}$ on 2 GPUs and $48\text{ ms}$ on 4 GPUs, matching our link-rate estimate within a millisecond. Both predictions of opposite sign hold: at batch 64, 2 GPUs are slower than 1 GPU and 4 GPUs gain less than $1.5\times$; at batch 512 in bf16, 4 GPUs reach roughly $3\times$ single-GPU throughput—the largest speedup of any single change in this section—with communication overlap pushing the measured speedup above the no-overlap floor the equation. (On an NVLink machine with $900\text{ GB/s}$ per direction, a $76\text{ MB}$ allreduce takes a fraction of a millisecond and scales nearly linearly even at batch 64.)

## Results

We now summarize all single-GPU optimizations side by side, verify that the final step still trains the model properly, and account for where its remaining time goes.

### Cumulative Comparison

The table below lists the single-GPU configurations in order. Each `step` ratio is measured relative to the row above it, `total` is the cumulative speedup over the baseline, and checkpointing appears below the line as an unkept branch. `TFLOP/s` converts throughput to arithmetic rate at $6\text{ FLOPs}$ per parameter and token, and `MFU` divides by the RTX 4090's $165.2\text{ TFLOP/s}$ bf16 peak (*model FLOPs utilization*, @chowdhery2022palm; Exercise 1 adds the attention-score FLOPs).

In [ ]:
flops = 6 * num_params  # Per token, without the attention scores
peak = {'tf32': 82.6e12, 'bf16': 165.2e12}  # FLOP/s of the RTX 4090
print(f'{"configuration":18}{"tokens/s":>10}{"step":>7}{"total":>7}'
      f'{"TFLOP/s":>9}{"MFU":>6}')
first = previous = next(iter(results.values()))
for name, speed in results.items():
    print(f'{name:18}{speed:10,.0f}{speed / previous:6.2f}x'
          f'{speed / first:6.2f}x{speed * flops / 1e12:9.1f}'
          f'{speed * flops / peak["bf16"]:6.0%}')
    previous = speed
print('-' * 57)
for name, speed in branches.items():  # Measured on top of the last row
    print(f'{name:18}{speed:10,.0f}{speed / previous:6.2f}x{"":7}'
          f'{speed * flops / 1e12:9.1f}{speed * flops / peak["bf16"]:6.0%}'
          '  (not kept)')

Cumulatively, the single-GPU optimizations double training throughput, reaching roughly 50% of the RTX 4090's bf16 peak. Nearly all of the gain comes from speeding up the matrix products (bf16) and reducing memory traffic around them (Inductor fusion in PyTorch, cuDNN fused attention in JAX), whereas removing host overhead and enlarging the minibatch contribute a few percent each, just as the profiles predicted.

Note that each speedup ratio depends on its position in the sequence (Exercise 7) and on the hardware balance: on a faster GPU (or with a smaller model), device execution is shorter, making host dispatch removal much more impactful.

### Training Check

Higher throughput is useless if an optimization silently breaks training. We verify correctness by training two identically initialized models—one with the baseline step and one with the final compiled bf16 step—on the same 200 minibatches of 512 windows ($\sim 13\text{M}$ tokens), using a 50-step linear learning-rate warm-up to prevent late AdamW loss spikes (that section). We check that both models agree within $0.15\text{ nats/byte}$ on the mean loss of the last 20 steps and on the validation loss (`d2l.lm_perplexity`, that section), and that both beat the $2.45\text{ nats/byte}$ bigram baseline of that section.

In [ ]:
learner = d2l.to_bf16(FusedGPT(len(data.vocab), num_hiddens=512,
                               rngs=nnx.Rngs(d2l.get_key())))
reference = d2l.GPT(len(data.vocab), num_hiddens=512,
                    rngs=nnx.Rngs(d2l.get_key()))
nnx.update(reference, nnx.state(learner, nnx.Param))  # One initialization
# The learning rate rises linearly to 1e-3 over the first 50 steps
warm = optax.adamw(optax.linear_schedule(1e-3 / 50, 1e-3, 49),
                   weight_decay=0.01)
reference_optimizer = nnx.Optimizer(reference, warm, wrt=nnx.Param)
steps = {'baseline': lambda X, Y: jit_step(reference, reference_optimizer,
                                           X, Y),
         'final': nnx.jit_partial(train_step, learner, nnx.Optimizer(
             learner, warm, wrt=nnx.Param), graph=False)}
batches = windows(corpus, 512)
minibatches = [next(batches) for _ in range(200)]  # The same for both
losses = {name: jnp.stack([step(X, Y) for X, Y in minibatches]).tolist()
          for name, step in steps.items()}
validation = {name: math.log(d2l.lm_perplexity(net, data.val_dataloader()))
              for name, net in (('baseline', reference), ('final', learner))}

In [ ]:
checkpoints = (1, 50, 100, 200)
late = {name: np.mean(curve[-20:]) for name, curve in losses.items()}
print(f'{"loss at step":14}' + ''.join(f'{i:>7}' for i in checkpoints)
      + f'{"mean of last 20":>17}{"validation":>12}')
for name, curve in losses.items():
    print(f'{name:14}' + ''.join(f'{curve[i - 1]:7.2f}' for i in checkpoints)
          + f'{late[name]:17.2f}{validation[name]:12.2f}')
    assert validation[name] < 2.45  # Better than the bigram model
for values in (late, validation):  # The final step trains like the baseline
    assert abs(values['baseline'] - values['final']) < 0.15

Both curves start at $\log 256 \approx 5.55$, track each other closely across all 200 steps, and reach nearly identical validation losses well below the bigram threshold. Notice, however, that 200 steps at batch size 512 ($13\text{M}$ tokens) end at a higher loss than 1,000 cosine-decayed steps at batch size 64 ($8\text{M}$ tokens) in that section: raw token throughput is not identical to time-to-target-loss (Exercise 5).

### Account of the Final Step

Finally, `account` normalizes the baseline and final profiles to milliseconds per $8{,}192$ tokens, comparing total wall-clock time against ideal peak arithmetic time, actual matrix-product kernel time, and non-matmul kernel time.

In [ ]:
def account(kernels, tokens, speed):
    """Milliseconds per 8,192 tokens of a step of `tokens` tokens that runs
    at `speed` tokens/s: the step, its matrix products, and all its other
    kernels."""
    per_step = 8192 / tokens  # Steps per 8,192 tokens
    products = per_step * sum(  # Kernel names as in d2l.show_profile
        ms for name, ms in kernels.items()
        if re.search(d2l.PRODUCT_KERNELS, name, re.I))
    other = per_step * sum(kernels.values()) - products
    return 1e3 * 8192 / speed, products, other

print(f'{"ms per 8,192 tokens":24}{"step":>6}{"at peak":>9}'
      f'{"products":>10}{"other":>7}')
for name, tokens, fmt in (('baseline', 64 * 128, 'tf32'),
                          ('+ batch 512', 512 * 128, 'bf16')):
    wall, products, other = account(profiles[name], tokens, results[name])
    arithmetic = 1e3 * 8192 * flops / peak[fmt]  # At the peak of the format
    print(f'{name + " (" + fmt + ")":24}{wall:6.1f}{arithmetic:9.1f}'
          f'{products:10.1f}{other:7.1f}')

Per $8{,}192$ tokens, the final step spends about $7\text{ ms}$ in matrix products (running at over 80% of the $5.7\text{ ms}$ bf16 ideal) and $5\text{ ms}$ in bandwidth-bound kernels between them—a classic roofline split where contractions perform over 99% of the FLOPs but take roughly 60% of the time [@Ivanov.Dryden.Ben-Nun.ea.2021]. Because neither faster arithmetic alone (e.g., fp8) nor eliminating non-matmul memory traffic alone can halve a step split $7\text{ ms}$ to $5\text{ ms}$, another $2\times$ speedup would require both.

## Summary

- A kernel profile classifies a training step's bottleneck and bounds the gain of each optimization before it is applied: reducing host dispatch saves at most the device idle time, kernel fusion saves at most the non-matmul device time, and doubling tensor-core precision throughput gives a speedup between $1/(1 - f/2)$ and $2$ when matrix products take a fraction $f$ of the step.
- Optimizations must be verified for numerical correctness and measured cumulatively, because removing one bottleneck shifts the remaining share of the step.
- On a single RTX 4090, our 19M-parameter GPT step doubled in speed (reaching ~50% MFU), driven almost entirely by bf16 matrix products and kernel fusion (Inductor / cuDNN FlashAttention).
- The data-parallel cost model $t_{\textrm{step}}(1) + t_{\textrm{allreduce}}(k)$ accurately predicted both a slowdown on 2 GPUs for the short fp32 step ($22\text{ ms}$ step vs. $31\text{ ms}$ PCIe allreduce) and a $\sim 3\times$ speedup on 4 GPUs for the longer batch-512 bf16 step ($90\text{--}100\text{ ms}$ step vs. $48\text{ ms}$ allreduce).

## Exercises

1. [code] **Model FLOPs utilization.** The GPT of that section has $L = 6$ blocks of width $d = 512$ and context length $T = 128$. Up to SwiGLU rounding, each block holds $12d^2$ weights in its matrix products, and training costs about $6\text{ FLOPs}$ per weight and token.
    1. The attention score product $\mathbf{Q}\mathbf{K}^\top$ and value aggregation are weightless matrix products. Count their training FLOPs per token and block (taking the backward pass as twice the forward pass and including masked entries). Show that their ratio to the weight FLOPs is $T/(6d)$, and evaluate this ratio for $T = 128$ and $T = 1{,}024$.
    1. Recompute the `TFLOP/s` and `MFU` columns of the cumulative table with attention FLOPs included, along with the fraction of each configuration's native format peak ($165\text{ TFLOP/s}$ in bf16, $83\text{ TFLOP/s}$ in tf32).
    1. Name two configurations or kernels in that section where theoretical model FLOPs differ from hardware-executed FLOPs, and state the direction of each difference.

1. [code] **Fixed cost of a step.** Model the bf16 step time at batch size $B$ windows as $t(B) = a + bB$, with fixed per-step cost $a$ and per-window cost $b$.
    1. Measure the step time at $B = 32, 64, 128, 256, 512$, fit $a$ and $b$ by least squares, and report the fraction of the $B = 64$ step accounted for by $a$.
    1. The text estimated AdamW updates and fp32/bf16 weight casts at $\sim 0.75\text{ ms/step}$. What fraction of $a$ does this explain? Identify other $B$-independent work and check against a profile at $B = 32$.
    1. Use your linear fit to predict throughput at $B = 1{,}024$ and $B = 2{,}048$, check whether they fit in $24\text{ GiB}$ of GPU memory, and measure those that do.

1. [code] **Cost of recomputation.** Activation checkpointing in that section beat both the FLOP-count estimate and the standalone forward-pass timing estimate.
    1. Profile the standalone forward pass and the checkpointed step with `d2l.kernel_times`. Match the kernels added by checkpointing against the standalone forward kernels: which reappear unchanged, and which are fused into backward kernels?
    1. Time the forward pass of the six blocks alone (excluding embeddings and the output layer) and recompute the throughput lower bound.
    1. Checkpoint only every second block: predict the throughput and memory from your measurements before running, then measure and compare.

1. [code] **Causality gate.** Suppose a defective attention kernel silently ignores the causal mask.
    1. Explain why the training loss, validation window loss, and the $2.45\text{ nats/byte}$ bigram threshold in that section would all fail to flag this defect if both baseline and learner used the defective kernel.
    1. Write a test that perturbs the final input token of every sequence in a minibatch and verifies that logits at all earlier positions remain unchanged within numerical tolerance. Run it on `learner` and on a copy whose attention omits the causal mask.

1. **Throughput versus time to target.** The model of that section estimates that training at batch size $b$ requires $E(b) = E_{\min}(1 + b / b_{\textrm{noise}})$ training examples to reach a target loss, where $b_{\textrm{noise}}$ is the gradient noise scale.
    1. Let $\rho > 1$ be the throughput ratio of batch 512 to batch 64. Show that batch 512 reaches the target loss in less wall-clock time if and only if
       $$b_{\textrm{noise}} > \frac{512 - 64\rho}{\rho - 1},$$
       and evaluate this threshold for $\rho = 1.2$, $\rho = 1.7$, and the measured $\rho$ from that section.
    1. Classify each change in that section, weak-scaling data parallelism, and switching to Muon (that section) by whether it alters $E(b)$, token throughput, or both.

1. [code] **Width 256.** At `num_hiddens=256`, `d2l.GPT` matches the architecture trained in that section.
    1. For the $8{,}192 \times 256$ by $256 \times 256$ output projection $\mathbf{W}_o$, compute the arithmetic intensity in 4-byte tf32 and 2-byte bf16, place both on the RTX 4090 roofline (the figure), and predict the bf16 speedup for this product.
    1. Run the baseline and the first two optimizations of that section at `num_hiddens=256`, profile them with `profile` and `account`, and compare the measured matrix-product speedup with your roofline prediction.

1. [code] **Order of the changes.** Each speedup ratio in the cumulative table depends on the optimizations applied before it.
    1. Using the baseline profile and the bound in that section, predict the speedup of applying bf16 directly to the uncompiled baseline.
    1. Measure that speedup and compare it with your prediction and with the `+ bf16` row in the cumulative table. Then apply compilation (`+ compile` / `+ jit_partial`) on top of bf16 and explain how both ratios shifted using the kernel profiles.

[Discussions](https://d2l.discourse.group/t/36793)